# Dataset Real Estate Listings (Data.World)

## Creating Folders

In [ ]:
!echo "Creating data folder"
!mkdir -p data/bronze data/silver data/gold

## Downloading Dataset

In [ ]:
import pandas as pd

In [ ]:
# df = pd.read_csv('https://query.data.world/s/3ox2v5trynvowrhc27fjqkcx3lytjf?dws=00000')
# df.to_csv("data/bronze/properati-BR-2016-11-01-properties-rent.csv", index=False)

In [ ]:
df = pd.read_csv('data/bronze/properati-BR-2016-11-01-properties-rent.csv')

In [ ]:
df.info()

In [ ]:
df.head()

## Cleaning (Silver)

### Parse string date to datetime

In [ ]:
def parse_date(src_df):
    dst_df = src_df.copy()
    dst_df['created_on'] = pd.to_datetime(src_df['created_on'], format="%Y-%m-%d")
    return dst_df

df_date = parse_date(df)
df_date['created_on'].info()

### Split 'place' column

In [ ]:
def split_place_with_parent_names(src_df):
    dst_df = src_df.copy()
    tmp_df = src_df['place_with_parent_names'].str.split("|", expand=True)

    dst_df['country'] = tmp_df[1]
    dst_df['region'] = tmp_df[2]
    dst_df['city'] = tmp_df[3]

    return dst_df

df_split = split_place_with_parent_names(df_date)
df_split

### Remove unused columns

In [ ]:
def remove_unused_colums(src_df):
    return src_df.drop(columns=[
        'place_name',
        'geonames_id',
        'place_with_parent_names',
        'lat-lon',
        'currency',
        'price_aprox_local_currency',
        'expenses',
        'properati_url',
        'description',
        'title',
        'image_thumbnail']
    )

df_less_columns = remove_unused_colums(df_split)
df_less_columns.info()

### Count # of NaN values

In [ ]:
df_less_columns.isna().sum()

In [ ]:
def replace_empty_values(src_df):
    dst_df = src_df.copy()
    # dst_df["geonames_id"] = src_df["geonames_id"].fillna(0)
    # dst_df["lat-lon"] = src_df["lat-lon"].fillna("")
    dst_df["lat"] = src_df["lat"].fillna(0)
    dst_df["lon"] = src_df["lon"].fillna(0)

    dst_df["price"] = src_df["price"].fillna(0)
    # dst_df["currency"] = src_df["currency"].fillna("")
    
    # dst_df["price_aprox_local_currency"] = src_df["price_aprox_local_currency"].fillna(0)
    dst_df["price_aprox_usd"] = src_df["price_aprox_usd"].fillna(0)
    dst_df["surface_total_in_m2"] = src_df["surface_total_in_m2"].fillna(0)
    dst_df["surface_covered_in_m2"] = src_df["surface_covered_in_m2"].fillna(0)
    dst_df["price_usd_per_m2"] = src_df["price_usd_per_m2"].fillna(0)
    dst_df["price_per_m2"] = src_df["price_per_m2"].fillna(0)

    dst_df["floor"] = src_df["floor"].fillna(0)
    dst_df["rooms"] = src_df["rooms"].fillna(0)

    # dst_df["expenses"] = src_df["expenses"].fillna(0)
    # dst_df["image_thumbnail"] = src_df["image_thumbnail"].fillna("")


    return dst_df

df_filled = replace_empty_values(df_less_columns)
df_filled.isna().sum()

### Drop entries without price

In [ ]:
def drop_entries_without_price(src_df):
    return src_df[src_df["price"] != 0].copy()

df_priced = drop_entries_without_price(df_filled)
df_priced

### Update USD values

In [ ]:
USD_TO_BRL = 5.00 # 1 USD = X BRL

def update_usd_values(src_df):
    dst_df = src_df.copy()

    usd_price = src_df['price'] / USD_TO_BRL
    dst_df['price_aprox_usd'] = usd_price

    valid_surface = src_df['surface_covered_in_m2'] > 0
    dst_df.loc[valid_surface, 'price_usd_per_m2'] = usd_price / src_df['surface_covered_in_m2']

    return dst_df

df_usd = update_usd_values(df_priced)
df_usd

In [ ]:
df_usd.info()

In [ ]:
df_usd.to_csv('data/silver/properati-BR-2016-11-01-properties-rent.csv', index=False)

## Analysis (Gold)

In [ ]:
import matplotlib.pyplot as plt

df_gold = pd.read_csv("data/silver/properati-BR-2016-11-01-properties-rent.csv")
df_gold['created_on'] = pd.to_datetime(df_gold['created_on'])
df_gold.info()

### Average Rent Price per Region

In [ ]:

avg_price_per_region = df_gold.groupby('region')['price_aprox_usd'].mean().sort_values(ascending=False)

plt.barh(avg_price_per_region.index, avg_price_per_region.values, color='skyblue', edgecolor='black')

plt.xlabel('Avg Price (U$)')
plt.ylabel('Regions')
plt.title('Average Renting Price per Region')

plt.show()

### Average Price per m2 per Region

In [ ]:
avg_price_m2_by_region = df_gold.groupby('region')['price_usd_per_m2'].mean().sort_values(ascending=True)

plt.barh(avg_price_m2_by_region.index, avg_price_m2_by_region.values, color='lightgreen', edgecolor='black')

plt.title('Average Price per m2 by Region', fontsize=14, fontweight='bold')
plt.xlabel('Average Price per m2 (U$)', fontsize=12)
plt.ylabel('Region', fontsize=12)
plt.tight_layout()
plt.show()

### Average Price per Type and Type distribution

In [ ]:


type_counts = df_gold['property_type'].value_counts()

avg_price_by_type = df_gold.groupby('property_type')['price_aprox_usd'].mean().sort_values(ascending=False)

colors = ['#ff9999', '#66b3ff', "#66ffe0", "#f0ff66"]  # Soft pink/red for House, Soft blue for Apartment

plt.figure(figsize=(8,5))
plt.subplot(1, 2, 1)
plt.pie(
    type_counts, 
    labels=type_counts.index, 
    autopct='%1.1f%%',       # Shows percentages rounded to 1 decimal place
    startangle=90,           # Rotates the start of the pie chart for clean alignment
    colors=colors, 
    wedgeprops={'edgecolor': 'black'} # Adds clean borders to pie slices
)
plt.title('Type Distribution', fontsize=12, fontweight='bold')

x_pos = [2, 4, 6, 8]

plt.subplot(1, 2, 2)
plt.bar(
    x_pos, 
    avg_price_by_type.values, 
    color=colors, 
    edgecolor='black',
)
plt.xticks(x_pos, avg_price_by_type.index)
plt.title('Avg Price by Type', fontsize=12, fontweight='bold')
plt.xlabel('Property Type')
plt.ylabel('Avg Price (U$)')
plt.subplots_adjust(wspace=0.4)

plt.tight_layout()
plt.show()

### Market price trends along the time (90 days window)

In [ ]:
trend_data = df_gold.groupby(['created_on', 'property_type'])['price_aprox_usd'].mean().unstack()

# Smooth out the daily transactional noise using a X-day rolling window
smooth_trends = trend_data.rolling(window=90, min_periods=1).mean()

# Plot the clear lines
plt.plot(smooth_trends.index, smooth_trends['house'], color='#ff9999', linewidth=2.5, label='Houses')
plt.plot(smooth_trends.index, smooth_trends['apartment'], color='#66b3ff', linewidth=2.5, label='Apartments')
plt.plot(smooth_trends.index, smooth_trends['store'], color="#66ffdb", linewidth=2.5, label='Store')
plt.plot(smooth_trends.index, smooth_trends['PH'], color="#f7ff66", linewidth=2.5, label='PH')

# Formatting enhancements
plt.title('Real Estate Market Price Trends (90 day window)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Smoothed Average Price ($)')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend()
plt.xticks(rotation=30, ha='right')

# plt.tight_layout()
plt.show()

In [ ]:
df_filtered = df_gold[
    (df_gold['rooms'].between(0, 25)) &
    (df_gold['region'] == "São Paulo") &
    (df_gold['property_type'] == "house")
    # (df_gold['price_aprox_usd'])
]

# This yields a number between -1 and 1. Closer to 1 means a strong positive relationship.
correlation = df_filtered['price_aprox_usd'].corr(df_filtered['rooms'])

# 4. Create the Scatter Plot
# alpha=0.6 makes dots semi-transparent so you can spot where multiple listings overlap
plt.scatter(
    df_filtered['rooms'], 
    df_filtered['price_aprox_usd'], 
    color='purple', 
    alpha=0.6, 
    edgecolors='black', 
    s=50  # Adjusts the size of the dots
)

# 5. Add Titles, Labels, and display the exact correlation metric dynamically
plt.title(f'Correlation Between Price & Rooms (R = {correlation:.2f})', fontsize=13, fontweight='bold')
plt.xlabel('Number of Rooms', fontsize=11)
plt.ylabel('Price ($)', fontsize=11)

# Clean up layout adjustments
plt.grid(True, linestyle='--', alpha=0.5) # A grid is highly recommended for scatter plots
plt.tight_layout()